In [1]:

import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
import scipy as sc
%matplotlib inline
#import seaborn as sns
#Make it so you can see all the columns when you print a table
pd.options.display.max_columns = 400
pd.options.display.max_rows = 40
from scipy import stats
import cmocean
from matplotlib import rcParams
import matplotlib.lines as mlines
import matplotlib
#import statsmodels.api as sm
import gsw
#from sklearn.decomposition import PCA
import xarray as xr
#import cmocean
import cartopy as cart
from cartopy import config
import cartopy.crs as ccrs

from xhistogram.xarray import histogram
#import sparse
plt.style.use('classic')
rcParams['font.family'] = 'sans-serif'

rcParams['font.sans-serif'] = ['Arial']
import pykrige.kriging_tools as kt
from pykrige.ok import OrdinaryKriging
from pykrige.uk import UniversalKriging
import gstools as gs


In [81]:
nd = pd.read_excel('lat lons for picarro sedi.xlsx')
c13_data = xr.open_dataset('GLODAPv2.2016b.OmegaC.nc')

print(c13_data.values)
print(nd.columns)

c13_data = c13_data.assign_coords(depth_surface=('depth_surface', c13_data.depth_surface.values))


<bound method Mapping.values of <xarray.Dataset>
Dimensions:        (lon: 360, lat: 180, depth_surface: 33, snr: 1)
Coordinates:
  * lon            (lon) float64 20.5 21.5 22.5 23.5 ... 376.5 377.5 378.5 379.5
  * lat            (lat) float64 -89.5 -88.5 -87.5 -86.5 ... 86.5 87.5 88.5 89.5
Dimensions without coordinates: depth_surface, snr
Data variables:
    OmegaC         (depth_surface, lat, lon) float64 ...
    OmegaC_error   (depth_surface, lat, lon) float64 ...
    Input_mean     (depth_surface, lat, lon) float64 ...
    Input_std      (depth_surface, lat, lon) float64 ...
    Input_N        (depth_surface, lat, lon) float64 ...
    OmegaC_relerr  (depth_surface, lat, lon) float64 ...
    SnR            (snr) float64 ...
    CL             (snr) float64 ...
    Depth          (depth_surface) float64 ...
Attributes:
    Description:          1 X 1 global mapped field of OmegaCa from the GLODA...
    Created:              Created by Siv K. Lauvset on 12-May-2016 18:37:27
    Instit

In [91]:
ndsurf1=nd
ndsurf1.loc[ndsurf1['lon'] > 180, 'lon'] = ndsurf1.loc[ndsurf1['lon'] > 180,'lon']-360

ndsurf1['nearestlat']=c13_data.lat.sel(lat=ndsurf1['lat'].values,method='nearest')

ndsurf1['nearestlon']=c13_data.lon.sel(lon=ndsurf1['lon'].values,method='nearest')

#ndsurf1['nearestdepth']=c13_data.depth_surface.sel(depth_surface=ndsurf1['depth'].values,method='nearest')

ndsurf1['c13']=np.zeros(ndsurf1.shape[0])

val = c13_data['OmegaC'].sel(depth_surface=ndsurf1.loc[l, 'nearestdepth'],lat=ndsurf1.loc[l, 'nearestlat'],lon=ndsurf1.loc[l, 'nearestlon'],method='nearest').values.item()
al = c13_data['OmegaC'].sel(depth_surface=ndsurf1.loc[l, 'nearestdepth'],lat=ndsurf1.loc[l, 'nearestlat'],lon=ndsurf1.loc[l, 'nearestlon'],method='nearest').values.item()
ndsurf1.loc[l, 'c13'] = val

for l in range(ndsurf1.shape[0]):
    ndsurf1['c13'][l]=(c13_data.sel(lat=ndsurf1['nearestlat'][l]).sel(lon=ndsurf1['nearestlon'][l])).values


/var/folders/dl/yffywxpd2fq46h16zbsq2v040000gn/T/ipykernel_21676/3484196942.py:17: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  ndsurf1['c13'][l]=(c13_data.sel(lat=ndsurf1['nearestlat'][l]).sel(lon=ndsurf1['nearestlon'][l])).values
/var/fol

In [99]:
ndsurf1['c13'] = np.nan

for l in range(ndsurf1.shape[0]):
    try:
        val = c13_data['OmegaC'].sel(
            lat=ndsurf1.loc[l, 'nearestlat'],
            lon=ndsurf1.loc[l, 'nearestlon'],
            method='nearest'
        ).values.item()
        ndsurf1.loc[l, 'c13'] = val
    except:
        print(f"Skipped row {l} due to selection error")
        continue
print(c13_data['OmegaC'])

Skipped row 0 due to selection error
Skipped row 1 due to selection error
Skipped row 2 due to selection error
Skipped row 3 due to selection error
Skipped row 4 due to selection error
Skipped row 5 due to selection error
Skipped row 6 due to selection error
Skipped row 7 due to selection error
Skipped row 8 due to selection error
Skipped row 9 due to selection error
Skipped row 10 due to selection error
Skipped row 11 due to selection error
<xarray.DataArray 'OmegaC' (depth_surface: 33, lat: 180, lon: 360)>
[2138400 values with dtype=float64]
Coordinates:
  * lon            (lon) float64 20.5 21.5 22.5 23.5 ... 376.5 377.5 378.5 379.5
  * lat            (lat) float64 -89.5 -88.5 -87.5 -86.5 ... 86.5 87.5 88.5 89.5
  * depth_surface  (depth_surface) int64 0 1 2 3 4 5 6 ... 26 27 28 29 30 31 32
Attributes:
    long_name:    calcite saturation state calculated at in situ temperature ...
    units:        
    Description:  The GLODAPv2.2016b input data were normalized to the year 2...
  